<a href="https://colab.research.google.com/github/thisismemayanka/analysis-projects/blob/main/5_Difference_in_difference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests pandas matplotlib seaborn statsmodels

In [ ]:
import requests
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf

from google.colab import userdata

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
API_KEY = userdata.get("openaq")

headers = {
    "X-API-Key": API_KEY
}

print("API Key Loaded Successfully")

API Key Loaded Successfully


In [ ]:
all_locations = []

page = 1

while True:

    params = {
        "country_id": 91,
        "limit": 1000,
        "page": page
    }

    response = requests.get(
        "https://api.openaq.org/v3/locations",
        headers=headers,
        params=params
    )

    response.raise_for_status()

    results = response.json()["results"]

    if len(results) == 0:
        break

    all_locations.extend(results)

    print(f"Downloaded Page {page} ({len(results)} locations)")

    page += 1

df_locations = pd.json_normalize(all_locations)

print("\nTotal Locations :", len(df_locations))

Downloaded Page 1 (1000 locations)
Downloaded Page 2 (1000 locations)
Downloaded Page 3 (1000 locations)
Downloaded Page 4 (1000 locations)
Downloaded Page 5 (1000 locations)
Downloaded Page 6 (1000 locations)
Downloaded Page 7 (1000 locations)
Downloaded Page 8 (1000 locations)
Downloaded Page 9 (1000 locations)
Downloaded Page 10 (1000 locations)
Downloaded Page 11 (1000 locations)
Downloaded Page 12 (1000 locations)
Downloaded Page 13 (1000 locations)
Downloaded Page 14 (1000 locations)
Downloaded Page 15 (1000 locations)
Downloaded Page 16 (1000 locations)
Downloaded Page 17 (1000 locations)
Downloaded Page 18 (1000 locations)
Downloaded Page 19 (1000 locations)
Downloaded Page 20 (1000 locations)
Downloaded Page 21 (1000 locations)
Downloaded Page 22 (1000 locations)
Downloaded Page 23 (1000 locations)
Downloaded Page 24 (1000 locations)
Downloaded Page 25 (900 locations)

Total Locations : 24900


In [ ]:
cities = [
    "Delhi",
    "New Delhi",
    "Noida",
    "Ghaziabad",
    "Faridabad",
    "Gurugram",
    "Gurgaon"
]

pattern = "|".join(cities)

stations = df_locations[
    df_locations["name"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

stations[[
    "id",
    "name",
    "coordinates.latitude",
    "coordinates.longitude"
]]

,id,name,coordinates.latitude,coordinates.longitude
10,13,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000
14,17,"R K Puram, Delhi - DPCC",28.563262,77.186937
41,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023
83,103,"Income Tax Office, Delhi - CPCB",28.623500,77.249400
180,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032
...,...,...,...,...
23776,6254665,"Commonwealth Sports Complex, Delhi - DPCC",28.615828,77.271992
23777,6254666,"IGNOU_Maidan Garhi, Delhi - DPCC",28.493624,77.201159
23789,6257818,"Cantonment Area, Delhi - DPCC",28.594169,77.125100
24111,6299678,"Ved Vihar-Loni, Ghaziabad - UPPCB",28.739153,77.273668


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
sensor_rows = []

for _, row in stations.iterrows():

    sensors = row["sensors"]

    if isinstance(sensors, list):

        for sensor in sensors:

            parameter = sensor.get("parameter", {}).get("name")

            if parameter == "pm25":

                sensor_rows.append({

                    "location_id": row["id"],

                    "station": row["name"],

                    "sensor_id": sensor["id"]

                })

sensor_df = pd.DataFrame(sensor_rows)

sensor_df

,location_id,station,sensor_id
0,13,"Delhi Technological University, Delhi - CPCB",13864
1,17,"R K Puram, Delhi - DPCC",35
2,17,"R K Puram, Delhi - DPCC",12234787
3,50,"Punjabi Bagh, Delhi - DPCC",396
4,50,"Punjabi Bagh, Delhi - DPCC",12234796
5,103,"Income Tax Office, Delhi - CPCB",13861
6,235,"Anand Vihar, New Delhi - DPCC",384
7,235,"Anand Vihar, New Delhi - DPCC",12235610
8,236,"Mandir Marg, Delhi - DPCC",388
9,301,"Vikas Sadan, Gurugram - HSPCB",1166


In [ ]:
import time

def download_sensor(sensor_id,
                    start_date,
                    end_date):

    all_data = []

    page = 1

    while True:

        url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"

        params = {

            "date_from": start_date,

            "date_to": end_date,

            "limit": 1000,

            "page": page

        }

        r = requests.get(
            url,
            headers=headers,
            params=params
        )

        r.raise_for_status()

        results = r.json()["results"]

        if len(results) == 0:
            break

        all_data.extend(results)

        print(
            f"Sensor {sensor_id} | "
            f"Page {page} | "
            f"{len(results)} observations"
        )

        page += 1

        time.sleep(0.2)

    if len(all_data)==0:
        return pd.DataFrame()

    return pd.json_normalize(all_data)

In [ ]:
sensor_df

,location_id,station,sensor_id
0,13,"Delhi Technological University, Delhi - CPCB",13864
1,17,"R K Puram, Delhi - DPCC",35
2,17,"R K Puram, Delhi - DPCC",12234787
3,50,"Punjabi Bagh, Delhi - DPCC",396
4,50,"Punjabi Bagh, Delhi - DPCC",12234796
5,103,"Income Tax Office, Delhi - CPCB",13861
6,235,"Anand Vihar, New Delhi - DPCC",384
7,235,"Anand Vihar, New Delhi - DPCC",12235610
8,236,"Mandir Marg, Delhi - DPCC",388
9,301,"Vikas Sadan, Gurugram - HSPCB",1166


In [ ]:
selected_sensors = sensor_df[
    sensor_df["station"].isin([
        "Anand Vihar, New Delhi - DPCC",
        "R K Puram, Delhi - DPCC",
        "Vikas Sadan, Gurugram - HSPCB"
    ])
]

selected_sensors

,location_id,station,sensor_id
1,17,"R K Puram, Delhi - DPCC",35
2,17,"R K Puram, Delhi - DPCC",12234787
6,235,"Anand Vihar, New Delhi - DPCC",384
7,235,"Anand Vihar, New Delhi - DPCC",12235610
9,301,"Vikas Sadan, Gurugram - HSPCB",1166
10,301,"Vikas Sadan, Gurugram - HSPCB",14258988
92,7044,"R K Puram, Delhi - DPCC",20411


In [ ]:
import os
import json

RAW_FOLDER = "raw_data"
CHECKPOINT_FOLDER = "checkpoints"

os.makedirs(RAW_FOLDER, exist_ok=True)
os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)

In [ ]:
import os
import json
import time
import requests
import pandas as pd


def download_sensor(
    sensor_id,
    station,
    location_id,
    start_date,
    end_date,
    max_retries=5
):

    csv_file = os.path.join(
        RAW_FOLDER,
        f"sensor_{sensor_id}.csv"
    )

    checkpoint_file = os.path.join(
        CHECKPOINT_FOLDER,
        f"sensor_{sensor_id}.json"
    )

    # Resume support
    start_page = 1

    if os.path.exists(checkpoint_file):

        with open(checkpoint_file, "r") as f:

            checkpoint = json.load(f)

            start_page = checkpoint["last_page"] + 1

            print(
                f"Resuming Sensor {sensor_id} from page {start_page}"
            )

    page = start_page

    while True:

        url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"

        params = {

            "date_from": start_date,

            "date_to": end_date,

            "limit": 1000,

            "page": page

        }

        success = False

        for attempt in range(max_retries):

            try:

                r = requests.get(
                    url,
                    headers=headers,
                    params=params,
                    timeout=90
                )

                r.raise_for_status()

                success = True

                break

            except Exception as e:

                print(
                    f"Retry {attempt+1} "
                    f"Page {page}"
                )

                time.sleep(5)

        if not success:

            print(
                f"Skipping page {page}"
            )

            break

        results = r.json()["results"]

        if len(results) == 0:

            print("Finished sensor.")

            break

        page_df = pd.json_normalize(results)

        page_df["station"] = station
        page_df["location_id"] = location_id
        page_df["sensor_id"] = sensor_id

        page_df.to_csv(

            csv_file,

            mode="a",

            index=False,

            header=not os.path.exists(csv_file)

        )

        with open(checkpoint_file, "w") as f:

            json.dump(
                {

                    "last_page": page

                },
                f
            )

        print(

            f"Sensor {sensor_id}"

            f" Page {page}"

            f" Saved {len(page_df)}"

        )

        page += 1

        time.sleep(1)

In [ ]:
for _, row in selected_sensors.iterrows():

    print("=" * 60)

    print(f"Downloading {row['station']}")

    try:

        download_sensor(

            sensor_id=row["sensor_id"],

            station=row["station"],

            location_id=row["location_id"],

            start_date="2023-10-04",

            end_date="2023-12-02"

        )

    except Exception as e:

        print(e)

        continue

Resuming Sensor 35 from page 18
Retry 1 Page 18
Retry 2 Page 18
Retry 3 Page 18
Retry 4 Page 18
Retry 5 Page 18
Skipping page 18
Sensor 12234787 Page 1 Saved 1000
Sensor 12234787 Page 2 Saved 1000
Sensor 12234787 Page 3 Saved 1000
Sensor 12234787 Page 4 Saved 1000
Sensor 12234787 Page 5 Saved 1000
Sensor 12234787 Page 6 Saved 1000
Sensor 12234787 Page 7 Saved 1000
Sensor 12234787 Page 8 Saved 1000
Sensor 12234787 Page 9 Saved 1000
Sensor 12234787 Page 10 Saved 1000
Sensor 12234787 Page 11 Saved 1000
Sensor 12234787 Page 12 Saved 1000
Sensor 12234787 Page 13 Saved 1000
Sensor 12234787 Page 14 Saved 1000
Sensor 12234787 Page 15 Saved 1000
Retry 1 Page 16
Sensor 12234787 Page 16 Saved 1000
Retry 1 Page 17
Retry 2 Page 17
Retry 3 Page 17
Retry 4 Page 17
Retry 5 Page 17
Skipping page 17
Sensor 12235610 Page 1 Saved 1000
Sensor 12235610 Page 2 Saved 1000
Sensor 12235610 Page 3 Saved 1000
Sensor 12235610 Page 4 Saved 1000
Sensor 12235610 Page 5 Saved 1000
Sensor 12235610 Page 6 Saved 1000
Sen

In [ ]:
import glob

files = glob.glob("raw_data/*.csv")

print(files)

['raw_data/sensor_1166.csv', 'raw_data/sensor_12235610.csv', 'raw_data/R_K_Puram_Delhi_-_DPCC_20411.csv', 'raw_data/Anand_Vihar_New_Delhi_-_DPCC_384.csv', 'raw_data/R_K_Puram_Delhi_-_DPCC_35.csv', 'raw_data/sensor_12234787.csv', 'raw_data/sensor_35.csv', 'raw_data/sensor_14258988.csv', 'raw_data/sensor_20411.csv', 'raw_data/Vikas_Sadan_Gurugram_-_HSPCB_14258988.csv', 'raw_data/R_K_Puram_Delhi_-_DPCC_12234787.csv', 'raw_data/sensor_384.csv', 'raw_data/Anand_Vihar_New_Delhi_-_DPCC_12235610.csv', 'raw_data/Vikas_Sadan_Gurugram_-_HSPCB_1166.csv']


In [ ]:
dfs = []

for file in files:

    print(file)

    dfs.append(
        pd.read_csv(file)
    )

master = pd.concat(
    dfs,
    ignore_index=True
)

master.to_csv(
    "all_measurements.csv",
    index=False
)

print(master.shape)

raw_data/sensor_1166.csv
raw_data/sensor_12235610.csv
raw_data/sensor_12234787.csv
raw_data/sensor_35.csv
raw_data/sensor_14258988.csv
raw_data/sensor_20411.csv
raw_data/sensor_384.csv
(95000, 27)


In [ ]:
import shutil

shutil.rmtree("checkpoints")